In [2]:
import torch
B,N,F = 2,4,3
x = torch.randn(B,N,F)
adj = torch.eye(N).unsqueeze(0).expand(B,-1,-1)+torch.diag(torch.ones(N-1),1).unsqueeze(0).expand(B,-1,-1)+torch.diag(torch.ones(N-1),-1).unsqueeze(0).expand(B,-1,-1)
deg = adj.sum(dim=-1,keepdim=True).clamp(min=1.0)
neighbor_mean = adj @ x /deg
print('Neighbor mean shape:',neighbor_mean.shape)
print(neighbor_mean[0,0])

Neighbor mean shape: torch.Size([2, 4, 3])
tensor([-0.7064, -0.6491,  0.1963])


In [3]:
import torch
B,N,F = 2,4,3
x = torch.randn(B,N,F)
neighbor_mean = torch.randn(B,N,F)
h = torch.cat([x,neighbor_mean],dim=-1)
print('After CONCAT shape:',h.shape)
print(f'特征维度应该是 2*F = {2*F}')

After CONCAT shape: torch.Size([2, 4, 6])
特征维度应该是 2*F = 6


In [6]:
import torch
import torch.nn.functional as F
B,N,F_in,F_out = 2,4,3,4
H=2
x = torch.randn(B,N,F_in)
W = torch.randn(F_in,H*F_out)*0.1
a_l = torch.randn(H,F_out,1)*0.1
a_r = torch.randn(H,F_out,1)*0.1
h = (x@W).view(B,N,H,F_out)
f_l = (h*a_l.view(1,1,H,F_out)).sum(dim=-1)
f_r = (h*a_r.view(1,1,H,F_out)).sum(dim=-1)
e = f_l.unsqueeze(2)+f_r.unsqueeze(1)
e = e.permute(0,3,1,2)
print(e.shape)
print(e[0,0])

torch.Size([2, 2, 4, 4])
tensor([[ 0.0024,  0.0090, -0.0128, -0.0045],
        [ 0.0142,  0.0208, -0.0009,  0.0074],
        [ 0.0536,  0.0602,  0.0384,  0.0467],
        [ 0.0213,  0.0278,  0.0061,  0.0144]])


In [7]:
import torch
import torch.nn.functional as F
B,H,N,_ = 2,2,4,4
e = torch.randn(B,H,N,N)
adj = torch.eye(N).unsqueeze(0).unsqueeze(0).expand(B,H,-1,-1)
mask = (adj>0)
e = torch.where(mask,e,torch.tensor(-1e9))
e = F.leaky_relu(e,negative_slope=0.2)
alpha = F.softmax(e,dim=-1)
print('Alpha shape:',alpha.shape)
print(alpha[0,0,0].sum().item())

Alpha shape: torch.Size([2, 2, 4, 4])
1.0


In [8]:
import torch
import torch.nn.functional as F
B,N,F_in,F_out = 2,4,3,5
H=4
x = torch.randn(B,N,F_in)
W = torch.randn(F_in,H*F_out)*0.1
a_l = torch.randn(H,F_out,1)*0.1
a_r = torch.randn(H,F_out,1)*0.1
adj = torch.eye(N).unsqueeze(0).unsqueeze(0).expand(B,H,-1,-1)
h = (x@W).view(B,N,H,F_out)
f_l = (h*a_l.view(1,1,H,F_out)).sum(dim=-1)
f_r = (h*a_r.view(1,1,H,F_out)).sum(dim=-1)
e = (f_l.unsqueeze(2)+f_r.unsqueeze(1)).permute(0,3,1,2)
e = torch.where(adj>0,e,torch.tensor(-1e9))
e = F.leaky_relu(e,0.2)
alpha = F.softmax(e,dim=-1)
h_perm = h.permute(0,2,1,3)
out_per_head = torch.einsum('bhjj,bhif->bhif',alpha,h_perm)
out = out_per_head.permute(0,2,1,3).contiguous().view(B,N,H*F_out)
print('Output shape:',out.shape)
print('期望: torch.Size([2, 4, 20])')

Output shape: torch.Size([2, 4, 20])
期望: torch.Size([2, 4, 20])


In [9]:
import torch
import torch.nn as nn
from torch_geometric.nn import global_add_pool

class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps=True):
        super().__init__()
        self.eps = nn.Parameter(torch.zeros(1)) if eps else 0
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU()
        )

    def forward(self, x, edge_index):
        # 聚合邻居信息
        out = self.mlp((1 + self.eps) * x + self.propagate(edge_index, x))
        return out

    def propagate(self, edge_index, x):
        # 简单实现：对每个目标节点求和邻居特征
        row, col = edge_index
        out = torch.zeros_like(x)
        out.scatter_add_(0, row.unsqueeze(-1).expand_as(x[col]), x[col])
        return out

class GIN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers=5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GINConv(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GINConv(hidden_dim, hidden_dim))

        # 读出后拼接所有层，维度为 in_dim + num_layers * hidden_dim
        self.fc = nn.Linear(in_dim + num_layers * hidden_dim, out_dim)

    def forward(self, x, edge_index, batch):
        h = x
        layer_reps = [h]  # 第 0 层
        for conv in self.convs:
            h = conv(h, edge_index)
            layer_reps.append(h)

        # 每层 sum pooling 后拼接
        h_g = torch.cat([global_add_pool(r, batch) for r in layer_reps], dim=-1)
        return self.fc(h_g)

d:\pythonprojects\practice-github\torch_final\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
